# An-Ra V4 — Canonical Shared-Vault T4 Trainer

This notebook continues the canonical 181M-parameter V4 model from the latest verified full-resume checkpoint. It uses one canonical writer, a signed launch contract, deterministic token windows, and Drive-backed checkpoint durability every 200 optimizer steps or 60 minutes.

**Before Run all:** select a T4 GPU runtime. The notebook searches the mounted account's `MyDrive`, canonical `MyDrive/AnRa/cluster`, Shared Drives, and Drive shortcut targets for the live `checkpoint-vault`, both data-pack parts, and signing key. For cross-account training, share the real folder with **Editor** access and add a shortcut to the Colab account's My Drive. A compressed `checkpoint-vault` file is rejected. Do not run two notebooks with `WORKER_ROLE = "canonical_trainer"` at the same time.

In [ ]:
# Operator configuration
WORKER_ROLE = "canonical_trainer"  # canonical_trainer or verify_only
WORKER_ID = "colab-t4-primary"
REPO_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
REPO_REF = "iterate500"
SESSION_BUDGET_MINUTES = 180
DRAIN_RESERVE_MINUTES = 30
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 8

DRIVE_MOUNT_ROOT = "/content/drive"
PACK_PARTS = [
    ("v4_phase_a_170m_seed1301.tar.gz.part00", 83886080, "9efe814598f52275dee15cb70e981e1bb375e24dbf97f39788aa4c84498f33f0"),
    ("v4_phase_a_170m_seed1301.tar.gz.part01", 63233323, "c073e325d2fbe09fe4afefe75c251db59540ea3358e34d8a184d7bb2831e0f6a"),
]
PACK_ARCHIVE_SHA256 = "07f01bf4809667acc670eb9c94dfab38d28522d7bfc2d4c930e71898cff86ee7"
BASELINE_RESUME = ("anra_v4_same_commit_interrupt_part2.pt", 2168037221, "837f7721dbef6862aef824bb8e9d642f54739252b02fab2a63e00797f0e8fb4b")
print({"role": WORKER_ROLE, "worker": WORKER_ID, "training_minutes": SESSION_BUDGET_MINUTES - DRAIN_RESERVE_MINUTES})

In [ ]:
# Mount the authorized account's Drive and enforce the requested accelerator.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, subprocess, torch
assert torch.cuda.is_available(), "No CUDA GPU. Select Runtime > Change runtime type > T4 GPU."
gpu_name = torch.cuda.get_device_name(0)
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert "T4" in gpu_name.upper(), f"Expected a T4 runtime, received {gpu_name}"
assert total_gib >= 14, f"T4 memory contract failed: {total_gib:.1f} GiB"
subprocess.run(["nvidia-smi"], check=True)
print(f"READY: {gpu_name}, {total_gib:.1f} GiB")

In [ ]:
# Clone a clean operational checkout. The signed launch records the exact commit.
import pathlib, shutil, subprocess, time
REPO = pathlib.Path('/content/anra')
os.chdir('/content')  # Never delete the process's current working directory.
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
clone_command = [
    'git', '-c', 'http.version=HTTP/1.1', 'clone',
    '--depth', '1', '--single-branch', '--branch', REPO_REF,
    REPO_URL, str(REPO),
]
for attempt in range(1, 4):
    if REPO.exists():
        shutil.rmtree(REPO)
    result = subprocess.run(
        clone_command,
        text=True,
        capture_output=True,
        env=git_env,
    )
    if result.returncode == 0:
        break
    print(f'GitHub clone attempt {attempt}/3 failed (exit {result.returncode}).')
    print(result.stderr.strip() or result.stdout.strip() or 'Git returned no diagnostic.')
    if attempt == 3:
        raise RuntimeError('Unable to clone the exact training branch after 3 attempts.')
    time.sleep(2 ** attempt)
os.chdir(REPO)
branch = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
assert branch == REPO_REF, f'Expected branch {REPO_REF}, received {branch}'
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert not subprocess.check_output(["git", "status", "--porcelain"], text=True).strip()
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", "."], check=True)
print(f"Clean source branch/commit: {branch}@{commit}")

In [ ]:
# Resolve shared/current-account assets, then reconstruct the immutable 170M-token window locally.
import hashlib, tarfile
from training.colab_shared_assets import resolve_colab_training_assets

def sha256_file(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(block_size), b''):
            digest.update(block)
    return digest.hexdigest()

SCRATCH = pathlib.Path('/content/anra-scratch')
SCRATCH.mkdir(parents=True, exist_ok=True)
try:
    ASSETS = resolve_colab_training_assets(DRIVE_MOUNT_ROOT)
except FileNotFoundError as error:
    if 'Missing training asset' not in str(error):
        raise
    print('A data part is outside the mounted shortcut; using authenticated Shared-with-me lookup.')
    ASSETS = resolve_colab_training_assets(DRIVE_MOUNT_ROOT, require_pack_parts=False)
VAULT_ROOT = str(ASSETS.vault_root)
PACK_PATHS = {path.name: path for path in ASSETS.pack_parts}
print(f"Shared training vault: {VAULT_ROOT} (step {ASSETS.vault_step})")

_drive_api = None
def download_shared_drive_file(name, expected_size=None, expected_hash=None, require_unique=False):
    global _drive_api
    if _drive_api is None:
        from google.colab import auth
        import google.auth, httplib2
        from google_auth_httplib2 import AuthorizedHttp
        from googleapiclient.discovery import build
        auth.authenticate_user()
        credentials, _ = google.auth.default()
        transport = AuthorizedHttp(credentials, http=httplib2.Http(timeout=120))
        _drive_api = build('drive', 'v3', http=transport, cache_discovery=False)
    escaped = name.replace('\\', '\\\\').replace("'", "\\'")
    response = _drive_api.files().list(
        q=f"name = '{escaped}' and trashed = false",
        spaces='drive', includeItemsFromAllDrives=True, supportsAllDrives=True,
        fields='files(id,name,size,modifiedTime),nextPageToken', pageSize=100,
    ).execute(num_retries=3)
    candidates = response.get('files', [])
    if expected_size is not None:
        candidates = [item for item in candidates if int(item.get('size', -1)) == expected_size]
    candidates.sort(key=lambda item: item.get('modifiedTime', ''), reverse=True)
    if require_unique and len(candidates) != 1:
        raise RuntimeError(f"Expected exactly one shared {name}, found {len(candidates)}")
    from googleapiclient.http import MediaIoBaseDownload
    rejected = []
    for item in candidates:
        destination = SCRATCH / name
        temporary_download = destination.with_suffix(destination.suffix + '.download')
        with temporary_download.open('wb') as handle:
            request = _drive_api.files().get_media(fileId=item['id'], supportsAllDrives=True)
            downloader = MediaIoBaseDownload(handle, request)
            done = False
            while not done:
                _, done = downloader.next_chunk()
        valid_size = expected_size is None or temporary_download.stat().st_size == expected_size
        actual_hash = sha256_file(temporary_download)
        valid_hash = expected_hash is None or actual_hash == expected_hash
        if valid_size and valid_hash:
            temporary_download.replace(destination)
            print(f"Verified shared Drive file: {name}")
            return destination
        rejected.append(f"id={item['id']} size={temporary_download.stat().st_size} sha256={actual_hash}")
        temporary_download.unlink(missing_ok=True)
    available_sizes = [item.get('size', 'unknown') for item in response.get('files', [])]
    raise FileNotFoundError(
        f"No valid Drive file matched {name}; API sizes={available_sizes}; rejected={rejected}"
    )

for name, expected_size, expected_hash in PACK_PARTS:
    mounted_part = PACK_PATHS.get(name)
    mounted_valid = bool(
        mounted_part and mounted_part.is_file()
        and mounted_part.stat().st_size == expected_size
        and sha256_file(mounted_part) == expected_hash
    )
    if not mounted_valid:
        if mounted_part:
            print(f"Mounted copy failed integrity validation: {mounted_part}; trying Drive versions.")
        PACK_PATHS[name] = download_shared_drive_file(name, expected_size, expected_hash)

archive = SCRATCH / 'v4_phase_a_170m_seed1301.tar.gz'
temporary = archive.with_suffix(archive.suffix + '.tmp')
with temporary.open('wb') as target:
    for name, expected_size, expected_hash in PACK_PARTS:
        part = PACK_PATHS[name]
        assert part.is_file(), f"Missing data pack part: {part}"
        assert part.stat().st_size == expected_size, f"Wrong size: {part}"
        assert sha256_file(part) == expected_hash, f"Corrupt data pack part: {part}"
        with part.open('rb') as source:
            shutil.copyfileobj(source, target, 8 * 1024 * 1024)
temporary.replace(archive)
assert sha256_file(archive) == PACK_ARCHIVE_SHA256, "Data archive hash mismatch"
pack_parent = REPO / 'output' / 'v2' / 'cloud_packs'
pack_parent.mkdir(parents=True, exist_ok=True)
with tarfile.open(archive, 'r:gz') as bundle:
    bundle.extractall(pack_parent, filter='data')
PACK_ROOT = pack_parent / 'v4_phase_a_170m_seed1301'
assert (PACK_ROOT / 'pack_manifest.json').is_file()
print(f"Verified data pack: {archive.stat().st_size:,} compressed bytes")

In [ ]:
# Materialize the newest full-resume checkpoint from the resolved shared vault.
import json

vault_root = pathlib.Path(VAULT_ROOT)
vault_pointer = vault_root / 'canonical.json'
resume_checkpoint = SCRATCH / 'resume-source.pt'
try:
    manifests = list((vault_root / 'manifests').glob('*.json'))
    assert manifests, f'No checkpoint manifests in {vault_root}'
    def manifest_step(path):
        try:
            candidate = json.loads(path.read_text())
            return int(candidate['lineage']['progress']['global_step'])
        except (OSError, KeyError, TypeError, ValueError, json.JSONDecodeError):
            return -1
    manifests.sort(key=manifest_step, reverse=True)
    selected = None
    rejected_manifests = []
    for manifest_path in manifests:
        try:
            candidate = json.loads(manifest_path.read_text())
            assert candidate['artifact_class'] == 'full_resume'
            assert candidate.get('resume_eligible') is True
            for expected_index, record in enumerate(candidate['chunks']):
                assert record['index'] == expected_index
                chunk = vault_root / 'chunks' / record['sha256'][:2] / f"{record['sha256']}.chunk"
                assert chunk.is_file(), f"missing chunk {record['index']}"
                assert chunk.stat().st_size == record['size_bytes'], f"wrong chunk size {record['index']}"
                assert sha256_file(chunk) == record['sha256'], f"corrupt chunk {record['index']}"
            selected = (manifest_path, candidate)
            break
        except (AssertionError, FileNotFoundError, KeyError, json.JSONDecodeError) as error:
            rejected_manifests.append(f"{manifest_path.name}: {error}")
    assert selected is not None, '; '.join(rejected_manifests[:5])
    manifest_path, manifest = selected
    expected_source = manifest['source']
    if not (resume_checkpoint.is_file() and resume_checkpoint.stat().st_size == expected_source['size_bytes'] and sha256_file(resume_checkpoint) == expected_source['sha256']):
        temporary = resume_checkpoint.with_suffix('.pt.tmp')
        with temporary.open('wb') as target:
            for record in manifest['chunks']:
                chunk = vault_root / 'chunks' / record['sha256'][:2] / f"{record['sha256']}.chunk"
                with chunk.open('rb') as source:
                    shutil.copyfileobj(source, target, 8 * 1024 * 1024)
        temporary.replace(resume_checkpoint)
    assert resume_checkpoint.stat().st_size == expected_source['size_bytes']
    assert sha256_file(resume_checkpoint) == expected_source['sha256']
    resume_step = int(manifest['lineage']['progress']['global_step'])
    source_label = f'newest complete Drive manifest {manifest_path.name}'
except (AssertionError, FileNotFoundError, KeyError, json.JSONDecodeError) as vault_error:
    resume_checkpoint.unlink(missing_ok=True)
    resume_checkpoint.with_suffix('.pt.tmp').unlink(missing_ok=True)
    print(f"Canonical vault is incomplete and will not be trusted: {vault_error}")
    baseline_name, baseline_size, baseline_hash = BASELINE_RESUME
    resume_checkpoint = download_shared_drive_file(baseline_name, baseline_size, baseline_hash)
    resume_step = 3
    source_label = 'verified local-rehearsal baseline fallback'
emergency_checkpoint = pathlib.Path(DRIVE_MOUNT_ROOT) / 'MyDrive' / 'anra-v4-emergency-step400.pt'
emergency_receipt = emergency_checkpoint.with_suffix('.pt.json')
if emergency_checkpoint.is_file() and emergency_receipt.is_file():
    emergency_proof = json.loads(emergency_receipt.read_text())
    assert emergency_checkpoint.stat().st_size == int(emergency_proof['size_bytes'])
    assert sha256_file(emergency_checkpoint) == emergency_proof['sha256']
    resume_checkpoint = emergency_checkpoint
    resume_step = int(emergency_proof.get('global_step', 400))
    source_label = 'verified emergency step-400 checkpoint'
verified_checkpoint_hash = sha256_file(resume_checkpoint)
print(f"Verified {source_label}: step={resume_step} sha256={verified_checkpoint_hash}")

In [ ]:
# Bind signing identity to the mounted checkpoint account, never a second API account.
import secrets
key_file = ASSETS.signing_key
legacy_key = pathlib.Path(DRIVE_MOUNT_ROOT) / 'MyDrive' / 'training-signing-keys.json'
recovery_key = pathlib.Path(DRIVE_MOUNT_ROOT) / 'MyDrive' / 'anra-v4-recovery-signing-keys.json'
if key_file is None and legacy_key.is_file():
    key_file = legacy_key
if key_file is None and recovery_key.is_file():
    key_file = recovery_key
if key_file is None:
    if resume_step != 3 or 'fallback' not in source_label:
        raise FileNotFoundError('No mounted campaign signing key; refusing to change identity on a healthy lineage')
    recovery_payload = {
        'schema_version': 1,
        'manifest': secrets.token_hex(32),
        'evidence': secrets.token_hex(32),
        'purpose': 'incomplete-vault recovery lineage',
    }
    temporary_key = recovery_key.with_suffix('.json.tmp')
    temporary_key.write_text(json.dumps(recovery_payload), encoding='utf-8')
    temporary_key.replace(recovery_key)
    key_file = recovery_key
    print('Created one persistent recovery signing identity in the mounted Drive account.')
private_keys = json.loads(key_file.read_text())
manifest_key = str(private_keys.get('manifest', ''))
evidence_key = str(private_keys.get('evidence', ''))
assert len(manifest_key) >= 64 and len(evidence_key) >= 64
os.environ['ANRA_MANIFEST_SIGNING_KEY'] = manifest_key
os.environ['ANRA_EVIDENCE_SIGNING_KEY'] = evidence_key
os.environ['ANRA_REQUIRE_SIGNED_EVIDENCE'] = '1'
print('Owner-private signing keys loaded without disclosure.')

In [ ]:
# Create and validate a launch bound to this commit, checkpoint, tokenizer, and remaining token window.
launch = REPO / 'output' / 'v2' / 'launch_manifests' / f'{WORKER_ID}.json'
artifact = SCRATCH / f'anra-v4-{WORKER_ID}.pt'
create_command = [
    'python', '-m', 'scripts.create_cloud_launch',
    '--pack-root', str(PACK_ROOT),
    '--output', str(launch),
    '--artifact-path', str(artifact),
    '--checkpoint-source', str(resume_checkpoint),
    '--worker-id', WORKER_ID,
    '--runtime-estimate-hours', str(SESSION_BUDGET_MINUTES / 60),
    '--batch-size', str(BATCH_SIZE),
    '--accumulation', str(GRADIENT_ACCUMULATION),
]
subprocess.run(create_command, check=True)
signed = json.loads(launch.read_text())
assert signed['git_commit'] == commit
print({
    'run_id': signed['run_id'],
    'commit': signed['git_commit'],
    'window': signed['token_window'],
    'checkpoint': signed['checkpoint_source_hash'],
})

In [ ]:
# Start the only canonical writer. New full-resume states are protected in Drive while training continues.
if WORKER_ROLE == 'verify_only':
    print('Verification complete. This worker will not modify canonical weights.')
else:
    assert WORKER_ROLE == 'canonical_trainer'
    pathlib.Path(VAULT_ROOT).mkdir(parents=True, exist_ok=True)
    os.environ['ANRA_DURABILITY_OUTBOX'] = str(SCRATCH / 'durability-outbox')
    os.environ['ANRA_DURABILITY_REPLICAS'] = json.dumps([
        {'name': 'drive-vault', 'path': VAULT_ROOT, 'kind': 'mounted_drive', 'canonical': True}
    ])
    os.environ['ANRA_DURABILITY_MIN_PROTECTED_REPLICAS'] = '1'
    os.environ['ANRA_DURABILITY_COPY_STREAMS'] = '2'
    os.environ['ANRA_DURABILITY_ACK_TIMEOUT_SECONDS'] = '1800'
    os.environ['ANRA_CHECKPOINT_EVERY_MIN'] = '60'
    os.environ['ANRA_DURABLE_CHECKPOINT_STEPS'] = '200'
    train_command = [
        'python', '-u', '-m', 'training.train_unified',
        '--mode', 'session',
        '--launch-manifest', str(launch),
        '--prepare_data', 'never',
        '--post-session-eval', 'none',
        '--data_path', 'training_data/anra_training.txt',
    ]
    process = subprocess.Popen(
        train_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=os.environ.copy(),
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'Trainer exited with status {return_code}; see streamed diagnostics above.')
    print('TRAINING SESSION COMPLETE. The final protected checkpoint is in checkpoint-vault.')